# Research domain tagging

**Recipe 37** · Level 3 (Intermediate) · Decision type: `Noul`

Assign independently judged research-domain labels to evidence passages while Python preserves provenance and enforces access controls.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S06: [TypeSafe AI: Cookbooks](https://docs.typesafe.ai/cookbooks)

## What you will build

Tag each evidence passage (a fabricated research abstract or excerpt) with every research domain it actually raises, from a fixed list of three: `biomedical`, `computational`, `economics`. Jev answers one independent `Noul` per domain, all three sent together in a single request because they describe the same passage, and Python turns the three probabilities into a business tag per domain, with a shared confidence gate sending an unconfident domain to an explicit review outcome instead of guessing. Before any of that, Python checks a passage's access classification: a passage marked `restricted` is filtered out before any state is built or any question is asked, so Jev never sees it at all. By the end you will have seen one passage's three answers up close, the two deterministic Python functions this recipe is built on — the access-control filter and the composition that turns three independent domain decisions into one task-level result — each proven on real fixtures and each covered by a test that fails when the corresponding rule is removed, and an evaluation that reports per-domain precision/recall/F1 (the "per question" view), the task-level share of passages tagged exactly right, and the access-control proof itself: a restricted passage is counted, but never sent and never appears in any tag, queue entry, or printed line.

One framing point worth stating plainly: this recipe is not an access-control system. The filter Python runs is a deterministic classification rule of this recipe's own fabricated scheme — a plain `if` over a field the fixtures carry — not a security feature, an authentication check, or a real embargo mechanism. What it demonstrates is narrower and still worth teaching: Python can enforce a rule like that *before* a request is built, so the model never sees text a policy says it should not see, and that enforcement does not depend on Jev at all.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays the stored answers in `fixtures/responses.json` offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls to the same three question definitions and changes nothing else. `run_header` states which mode ran, and every number below carries that mode with it.

Candidates are split into `accessible` and `restricted` immediately, before any state is built or any question is asked: `scored`, the sample this run's numbers are about, is `accessible` only, because a restricted passage is never decided and so never contributes a number.

In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    evaluate_outcomes,
    evaluate_selective,
    exact_set_match,
    multilabel_from_noul,
    multilabel_metrics,
    pool_label_decisions,
    pool_label_outcomes,
    select_confidence_threshold,
    select_threshold,
    selective_curve,
    threshold_sweep,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ReviewQueue
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_risk_coverage,
    plot_threshold_sweep,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
DOMAINS = list(helpers.DOMAINS)
examples = load_inputs()
by_id = {e.id: e for e in examples}
labels = load_labels()  # {id: sorted list of gold domains, [] when none apply}
# The one backend this notebook ever builds. Wrapped immediately in helpers.CountingBackend,
# which forwards every attribute (mode, model, recorded_dates) a plain backend has, so every
# request this notebook ever makes -- the ones below, and every later one -- is recorded from
# the start; "Evaluation" reports from this one running count, rather than building a second
# backend (which, under JEV_COOKBOOK_LIVE=1, would be a second LiveBackend spending a second,
# separate request budget against the same questions).
backend = helpers.CountingBackend(get_backend(fixtures=responses_path()))
questions = helpers.build_questions()

# candidates: every example with a gold label (excludes the two demo examples). The
# access-control filter runs on this list before anything else, so restricted passages never
# reach a question or a request.
candidates = [e for e in examples if e.split not in ("train", "demo")]
accessible = helpers.accessible_examples(candidates)
restricted = helpers.restricted_examples(candidates)

# N for the header is the number of examples this run's metrics are actually about: the
# accessible, scored ones. A restricted passage carries a gold label too (fixtures.md requires
# one for every validation/test example), but it is never decided, so it is not part of
# `scored` and contributes nothing below.
scored = accessible

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: "a number from the validation split is a selection step, not a
# result." This label does not depend on the run mode, unlike `check` above: even a recorded
# or live run must not print a bare validation number as if it were a reported result.
selection = " (a selection step, not a reported result)"

# Every accessible example describes the same passage with all three domains, so it needs
# exactly one request (see "The questions"), never three. This cache makes sure a passage
# shown individually and later scored in its split is decided once.
_decided = {}


def decide(example):
    if example.id not in _decided:
        _decided[example.id] = backend.decide(helpers.build_state(example.fields), questions)
    return _decided[example.id]


run_header(37, "Research domain tagging", backend=backend, n_examples=len(scored))

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl` carries `fields`: a `passage_id`, the passage `text`, and an `access` classification (`open` or `restricted`). `build_state` passes on the text only; the id and the access classification both stay in Python, so neither one passes through the model, and every result still traces back to its passage.

In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
example = demo["d01-all-demo"]
print("Python has:")
print(json.dumps(example.fields, indent=2))
state = helpers.build_state(example.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

## The questions

Three independent `Noul` propositions, one per domain in `helpers.DOMAINS`, built by a single Python loop over that fixed list. Each proposition is a statement that can be true or false ("The passage reports ..."), not a question, as CONTRIBUTING.md requires. A `Choice` would not fit here: `Choice` commits to exactly one option, and these three outcomes are not exclusive — a passage can report biomedical, computational, and economic work all at once, or none of the three, which is exactly what this recipe's use case asks for ("independently judged research-domain labels").

All three questions describe the same passage text, so `backend.decide` sends them together in a single request: TypeSafe's primitives page (S02) says independent questions over the same state belong in one request. A `Noul` has no confidence field at all (S03): the probability of yes, `noul`, is the only number it returns, so every threshold below — one business threshold per domain, and one shared confidence gate — is chosen from examples, never read directly off the answer.

In [ ]:
for domain, question in questions.items():
    print(f"{domain}:")
    print(f"  instructions: {question.instructions}")
    for side, text in question.criteria.items():
        print(f"    {side}: {text}")

## One answer up close

Before any aggregate number, look at three passages across the range this recipe covers: one that raises all three domains at once, one that raises none of them despite using look-alike vocabulary, and one that fools the stored answer into a confident, wrong domain tag. Each domain's answer holds one field of its own, `noul`, the probability that its proposition is true, plus a [`provenance`](../../docs/glossary.md#provenance) every answer carries (explained below); a `Noul` has no separate [`confidence`](../../docs/glossary.md#confidence) (S03), so how sure an answer is comes from how far `noul` sits from 0.5, not from a number printed on the answer itself.

In [ ]:
all_example = demo["d01-all-demo"]
all_answer = decide(all_example)
print(all_example.fields["text"])
for domain in DOMAINS:
    print(f"{domain}:")
    show_answer(all_answer[domain])
plot_answer_probabilities(
    {domain: all_answer[domain].noul for domain in DOMAINS},
    title="Domain probabilities: all three domains at once",
)

All three probabilities sit well above 0.5: this passage pairs a machine-learning readmission-risk model with a cost-accounting framework, which is genuinely biomedical and computational and economic at once, and its [gold label](../../docs/glossary.md#gold-label) set — the domains a person has determined the passage actually raises, used later in "Evaluation" — is all three. A single three-way `Choice` could only ever report one of these three; three independent `Noul` questions report all three at once, which is the entire reason this recipe does not use `Choice` ("The questions" above). `Provenance: synthetic` on every line above says where the answer came from: here always `synthetic`, since nothing in this recipe's fixtures came from a real call; a `recorded` answer would instead name the model the API returned and the date it was captured.

In [ ]:
none_example = by_id["v02-none-lookalike"]
none_answer = decide(none_example)
print(none_example.fields["text"])
for domain in DOMAINS:
    print(f"{domain}:")
    show_answer(none_answer[domain])
plot_answer_probabilities(
    {domain: none_answer[domain].noul for domain in DOMAINS},
    title="Domain probabilities: no research domain raised",
)

This passage describes a clinic's appointment-scheduling software — it uses "software" and "patients" and "budget" in passing, but it is an ordinary operations note, not a biomedical, computational, or economic research finding, and its gold label set is empty. Two empty sets match (`exact_set_match`'s definition, used below), so getting this passage exactly right means tagging nothing, not nothing at all. Every probability above is well under 0.5: this benign look-alike (CONTRIBUTING.md section 5 — a passage using a domain's vocabulary without its subject actually belonging to that domain) does not fool this particular stored answer, even though a very similar look-alike does, shown next.

In [ ]:
lookalike_example = by_id["v04-biomedical-lookalike"]
lookalike_answer = decide(lookalike_example)
print(lookalike_example.fields["text"])
for domain in DOMAINS:
    print(f"{domain}:")
    show_answer(lookalike_answer[domain])
plot_answer_probabilities(
    {domain: lookalike_answer[domain].noul for domain in DOMAINS},
    title="Domain probabilities: a benign look-alike, confidently wrong",
)

This passage describes the clinical course of a rare autoimmune skin condition — genuinely biomedical, and its gold label set is `{biomedical}` alone. It mentions that patients reported the condition's "daily cost" of managing flare-ups, but explicitly offers no economic analysis of that cost. The stored `economics` probability is nonetheless 0.83, well above 0.5 and, as "Python's part" below shows, above the frozen `economics` threshold too: a word associated with a domain, used without the passage actually raising that domain's research question, is exactly the look-alike case CONTRIBUTING.md section 5 names, and here it produces a genuinely wrong, confident tag rather than a near-miss. "Evaluation" below shows what that costs.

## Python's part

This recipe asks Python to do two separate jobs, and keeps them as two separate, tested functions in `helpers.py` rather than one tangled rule.

- **Filter passages by access control, before any request.** `helpers.accessible_examples` is the one place this recipe decides which passages may be sent at all, and nothing downstream of it — no state, no question, no request — ever sees a passage it excludes.
- **Compose three independent domain decisions into one task-level result.** `helpers.tag_domains` compares each domain's Noul answer against that domain's own threshold (the business tag) and separately checks a shared [confidence gate](../../docs/glossary.md#confidence-gate) built from `|2p - 1|` (S03, `jev_cookbook.evaluation.noul_confidence`); `helpers.compose` then turns the three per-domain decisions into one `PassageResult` — the domain set tagged, the domains sent to `review` instead, and the provenance of every one of those decisions, carried through rather than discarded. A domain below the confidence gate is not reported as present or absent, whichever side of its threshold the probability falls on: it is routed to [`review`](../../docs/glossary.md#review) (CONTRIBUTING.md section 4, "Uncertain or inconsistent results go to an explicit review outcome."). Both functions hold whatever the model answers, each checked by a test for every domain and every outcome in `tests/test_helpers.py`, not a sentence of prose — including a test, for each function, that fails when the function's own rule is removed.

### Access control

Every candidate example — every `validation` and `test` passage, before any split-specific work happens — is partitioned once, here, into `accessible` and `restricted`. The partition depends only on each example's `access` field; it has no way to depend on a passage's research-domain content, because `accessible_examples` never reads that content at all.

In [ ]:
print(f"candidates: {len(candidates)}{check}")
print(f"accessible (Python will send these): {len(accessible)}{check}")
print(f"restricted (Python will never send these): {len(restricted)}{check}")
for example in restricted:
    print(f"  withheld: {example.id}  gold domains: {labels[example.id]}")

Access is authored independently of a passage's research content: `v17-restricted`'s gold domains are `{biomedical}` and `v18-restricted`'s are `{computational, economics}`, so being restricted is not predictable from what a passage is about, and neither is being accessible. The demo passage shown below makes the filter concrete: Python never calls `helpers.build_state` on it, never builds a question about it, and never calls `backend.decide` on it — there is nothing to show "Jev sees" for this passage, because nothing is ever sent.

In [ ]:
restricted_demo = demo["d02-restricted-demo"]
print("Python has:")
print(json.dumps(restricted_demo.fields, indent=2))
print(f"accessible? {helpers.is_accessible(restricted_demo.fields['access'])}")
print("Jev sees: nothing -- Python declines to send this passage.")

Every accessible candidate is decided exactly once, through `helpers.decide_passages` -- the one function in this recipe that turns a list of examples into actual requests. It shares `_decided`, the same cache `decide()` uses above, so `v02-none-lookalike` and `v04-biomedical-lookalike` -- already decided individually in "One answer up close" -- are not asked again here: this call only sends a new request for an accessible candidate not already in the cache. `backend` itself is the `helpers.CountingBackend` wrapped in "Setup and run mode", so every one of those new requests is also recorded in its running `sent_states`, which "Evaluation" below reads rather than re-measuring with a second backend.

In [ ]:
decisions = helpers.decide_passages(candidates, backend, questions, cache=_decided)
print(f"decided accessible passages: {len(decisions)}{check}")
print(f"one entry per accessible passage: {len(decisions) == len(accessible)}{check}")
print(
    f"a restricted id ever appears among them: {bool(set(decisions) & {e.id for e in restricted})}{check}"
)

### Thresholds and the confidence gate

Two settings are chosen here, both on `validation`, and both frozen before `test` is touched.

- **One [business threshold](../../docs/glossary.md#business-threshold) per domain.** The three domains are different enough -- a passage that only brushes against "cost" language reads very differently from one that genuinely builds an economic model -- that one shared cut-off would fit some of them badly. Each is chosen with `select_threshold(objective="f1")` on that domain's own 18 `validation` examples.
- **One confidence gate, shared across all three domains.** Confidence sits on the same 0-to-1 scale for every domain (S03: the Noul formula is the Choice formula applied to a yes-or-no choice), so pooling every (passage, domain) pair in `validation` -- 18 passages times three domains, 54 pairs -- gives one usable sample for that one choice instead of three smaller ones.

Pooling has a real cost, worth naming rather than only defending. `docs/evaluation.md`'s own caution about `noul_confidence` bites here: `|2p - 1|` is distance from 0.5, not margin at a domain's own threshold, so when a threshold sits far from 0.5 -- every domain's does, around 0.8 -- an answer can read as confident while still sitting close to the line that actually decides it. A gate pooled across all three domains has no way to single out which domain's margin problem is worse than another's; it only ever sees one shared confidence number. "Evaluation" below shows what that costs on `test`.

In [ ]:
val_examples = [e for e in accessible if e.split == "validation"]
test_examples = [e for e in accessible if e.split == "test"]

val_gold = {e.id: set(labels[e.id]) for e in val_examples}
test_gold = {e.id: set(labels[e.id]) for e in test_examples}

val_noul_by_domain = {d: [decisions[e.id][d].noul for e in val_examples] for d in DOMAINS}
val_gold_by_domain = {d: [d in val_gold[e.id] for e in val_examples] for d in DOMAINS}

thresholds = {
    d: select_threshold(val_gold_by_domain[d], val_noul_by_domain[d], objective="f1")
    for d in DOMAINS
}
for d in DOMAINS:
    print(f"{d:14s} threshold {thresholds[d]:.4f}{selection}{check}")

`threshold_sweep` tries every distinct probability observed on `validation` for one domain and reports precision, recall and F1 at each; `select_threshold` picks the one with the highest F1, which is what the printed thresholds above come from, run once per domain. The chart below shows that sweep for `economics`, the domain this fixture set makes hardest: several look-alike passages store a probability just above or below the line, so no threshold separates `validation`'s economics examples perfectly.

In [ ]:
economics_sweep = threshold_sweep(val_gold_by_domain["economics"], val_noul_by_domain["economics"])
for point in economics_sweep:
    print(
        f"economics validation threshold {point.threshold:.4f}  precision {point.precision:.4f}  "
        f"recall {point.recall:.4f}  f1 {point.f1:.4f}{selection}{check}"
    )
plot_threshold_sweep(
    economics_sweep,
    chosen=thresholds["economics"],
    title=f"economics threshold sweep on validation (chosen {thresholds['economics']:.4f}){check}",
)

With every domain's business threshold frozen, the confidence gate is chosen next, pooling all 54 validation (passage, domain) pairs together. `select_confidence_threshold` needs to know, for each pair, whether the business rule (`noul >= threshold`) got that pair right, and how confident the answer was (`noul_confidence`); it then picks the gate with the highest [accuracy](../../docs/glossary.md#accuracy) among cut-offs whose answered subset covers at least `min_coverage` of the pairs, ties going to the lower threshold. A target coverage of 0.70, rather than a target accuracy of 1.0, is used here: tagging a research domain wrong is a low-stakes mistake compared to, say, approving a refund (S03's own example of a high-stakes action needing a stricter gate), and holding out for perfect accuracy on only 54 pooled pairs would mean trusting almost nothing automatically.

Before freezing this gate, the confidence page's own caution (S03) is worth checking: confidence is only worth gating on if answers tend to get *less* accurate as the gate falls. The chart below plots [risk](../../docs/glossary.md#risk) against [coverage](../../docs/glossary.md#coverage), on `validation`, before the cutoff is frozen.

In [ ]:
val_correct, val_confidence, _ = pool_label_decisions(
    [e.id for e in val_examples], DOMAINS, val_gold_by_domain, val_noul_by_domain, thresholds
)
print(f"pooled validation (passage, domain) pairs: {len(val_correct)}{selection}{check}")

val_curve = selective_curve(val_correct, val_confidence)
for t, coverage, acc, risk in zip(
    val_curve.thresholds, val_curve.coverage, val_curve.accuracy, val_curve.risk, strict=True
):
    print(
        f"validation gate >= {t:.4f}  coverage {coverage:.4f}  accuracy {acc:.4f}  "
        f"risk {risk:.4f}{selection}{check}"
    )
plot_risk_coverage(
    val_curve, label="validation, pooled", title=f"Risk vs. coverage on validation{check}"
)

In [ ]:
confidence_gate = select_confidence_threshold(val_correct, val_confidence, min_coverage=0.70)
print(f"chosen confidence gate, frozen from here on: {confidence_gate:.4f}{selection}{check}")

### Applying the rule

Applying both frozen settings to `v06-computational-lookalike` -- a passage describing an operating-room scheduling algorithm, genuinely `computational` and not `economics` or `biomedical` -- shows all three outcomes `tag_domains` can produce on one passage at once: a confident `yes` (`computational`), a confident `no` (`economics`), and a `review` (`biomedical`, stored at 0.46 -- a *correct* `no` by the business rule, since gold `biomedical` is also false here, but one barely more sure than an even split, so the confidence gate still withholds it from being reported as present or absent at all).

In [ ]:
apply_example = by_id["v06-computational-lookalike"]
apply_answer = decisions[apply_example.id]
apply_decisions = helpers.tag_domains(apply_answer, thresholds, confidence_gate)
for d in DOMAINS:
    dec = apply_decisions[d]
    print(
        f"{d:14s} noul {dec.noul:.2f}  tag {dec.tag}  outcome {dec.outcome:7s}  "
        f"reason: {dec.reason}"
    )
result = helpers.compose(apply_example.id, apply_decisions)
print(f"tags: {sorted(result.tags)}  review: {list(result.review)}")

Every domain whose outcome is `review`, for every accessible candidate in either split, goes into a `jev_cookbook.simulation.ReviewQueue`: a simulated queue that records what was flagged and why, and does nothing else with it (CONTRIBUTING.md section 4, "Actions are simulated."). `results` below holds every accessible passage's full `PassageResult`, built once and reused rather than recomputed, in "Evaluation". No restricted passage's id can appear in this queue: the loop below only ever iterates `accessible`, which, by construction, never contains one.

In [ ]:
all_decisions = {
    e.id: helpers.tag_domains(decisions[e.id], thresholds, confidence_gate) for e in accessible
}
results = {e.id: helpers.compose(e.id, all_decisions[e.id]) for e in accessible}

review_queue = ReviewQueue()
for e in accessible:
    result = results[e.id]
    for d in result.review:
        review_queue.submit(
            {"split": e.split, "passage_id": e.fields["passage_id"], "domain": d},
            all_decisions[e.id][d].reason,
            answer=decisions[e.id][d],
        )

val_queue_items = [item for item in review_queue.pending() if item.item["split"] == "validation"]
test_queue_items = [item for item in review_queue.pending() if item.item["split"] == "test"]
print(f"items sent to review, validation: {len(val_queue_items)}{selection}")
print(f"items sent to review, test: {len(test_queue_items)}{check}")
for item in review_queue.pending():
    label = selection if item.item["split"] == "validation" else check
    print(f"{item.item} - {item.reason}{label}")

## Evaluation

Both frozen settings were chosen above, on `validation`. This section reports what they do, in the two ways CONTRIBUTING.md's Level 3 scope asks a multi-stage recipe for, plus the access-control proof the use case names as its own acceptance criterion.

- **Per question**: per-domain precision, recall and F1 (the business rule alone, `noul >= threshold`, ignoring the confidence gate -- exactly as `jev_cookbook.evaluation.multilabel_from_noul` applies the frozen thresholds), their micro and macro summaries, and the confidence gate's own coverage, accuracy and risk, counted from `tag_domains`'s three outcomes with `pool_label_outcomes` fed to `evaluate_outcomes`.
- **Task level**: the share of passages whose *whole* composed domain set exactly matches its gold set (`exact_set_match`), which a per-domain number cannot show by itself -- a passage can get two of three domains right and still be wrong as a passage.
- **The access-control test**: a restricted passage is never sent, proven by instrumenting the actual request path rather than inferring it from where the filter sits in the code, and never appears in any tag, review entry, or printed line, proven directly from the committed decisions and queue above.

Nothing below reimplements `noul >= threshold` or the confidence check by hand: every number traces back to `tag_domains`, `compose`, or the shared `jev_cookbook.evaluation` functions already called above. **Terms, as used below:** [*precision*](../../docs/glossary.md#precision) is, of the passages the business rule tagged with a domain, the fraction that really raise it; [*recall*](../../docs/glossary.md#recall) is, of the passages that really raise a domain, the fraction the rule tagged; [*f1*](../../docs/glossary.md#f1) is their harmonic mean, low whenever either one is; [*support*](../../docs/glossary.md#support) is how many `test` passages really raise that domain; *micro* pools every domain's counts together before computing one precision, recall and F1, so a frequent domain dominates; *macro* averages each domain's own score equally; *coverage* is the fraction of (passage, domain) pairs the confidence gate answered rather than sent to review; *risk* is `1 - accuracy` among those it answered.

In [ ]:
val_noul_by_domain_eval = val_noul_by_domain  # already built in "Python's part"
val_predicted = multilabel_from_noul(val_noul_by_domain_eval, thresholds)
val_gold_sets = [frozenset(val_gold[e.id]) for e in val_examples]
val_metrics = multilabel_metrics(val_gold_sets, val_predicted, labels=DOMAINS)
for d in DOMAINS:
    counts = val_metrics["per_label"][d]
    print(
        f"validation {d:14s} precision {counts.precision:.3f}  recall {counts.recall:.3f}  "
        f"f1 {counts.f1:.3f}  support {counts.support}{selection}{check}"
    )

test_noul_by_domain = {d: [decisions[e.id][d].noul for e in test_examples] for d in DOMAINS}
test_predicted = multilabel_from_noul(test_noul_by_domain, thresholds)
test_gold_sets = [frozenset(test_gold[e.id]) for e in test_examples]
test_metrics = multilabel_metrics(test_gold_sets, test_predicted, labels=DOMAINS)
for d in DOMAINS:
    counts = test_metrics["per_label"][d]
    print(
        f"{d:14s} precision {counts.precision:.3f}  recall {counts.recall:.3f}  "
        f"f1 {counts.f1:.3f}  support {counts.support}{check}"
    )
print(
    f"micro  precision {test_metrics['micro']['precision']:.3f}  "
    f"recall {test_metrics['micro']['recall']:.3f}  f1 {test_metrics['micro']['f1']:.3f}{check}"
)
print(
    f"macro  precision {test_metrics['macro']['precision']:.3f}  "
    f"recall {test_metrics['macro']['recall']:.3f}  f1 {test_metrics['macro']['f1']:.3f}{check}"
)

No domain's `test` F1 is 1.0000: each of the three has at least one real, committed mistake behind it, so each number would read differently if that row's stored probability had landed on the other side of its domain's line. `biomedical` and `computational` each read precision 0.889, recall 1.000, f1 0.941: each has exactly one false positive and no false negative. `biomedical`'s false positive is `t19-economics-biomedical-wrong` (gold `{economics}`, stored `biomedical` 0.84, above the 0.8000 threshold); `computational`'s is `t20-economics-computational-wrong` (gold `{economics}`, stored `computational` 0.84, above the 0.7900 threshold) -- the two deliberately wrong look-alikes built for exactly this purpose. `economics` reads lowest, precision 0.889, recall 0.800, f1 0.842: one false positive (`t04-biomedical-lookalike`, stored `economics` 0.80, exactly at its threshold, despite a gold set of `{biomedical}` alone) and two false negatives, each one or two hundredths below the 0.8000 threshold (`t14-compecon-lookalike` at 0.79, `t16-all-lookalike` at 0.78, both genuinely `economics` by gold). None of these five numbers is unfalsifiable by construction: flipping any one of these five specific stored probabilities to the other side of its domain's threshold would change that domain's own F1. `tests/test_helpers.py::test_correcting_a_real_wrong_answer_changes_its_domains_f1` establishes exactly this, by actually recomputing F1 before and after the correction, for three of the five (`t19`'s `biomedical`, `t20`'s `computational`, `t04`'s `economics`); `t14` and `t16` move `economics`'s recall term the same way, for the same reason.

### Task level

`exact_set_match` scores a passage right only when its *entire* composed domain set matches gold -- tagging `{biomedical}` for a passage that should be `{biomedical, economics}` counts as wrong here even though two of the two decisions it actually made (biomedical: yes; nothing else offered) were each individually defensible reads of a partial set. This is the number a per-domain F1 cannot show by itself.

In [ ]:
val_exact = exact_set_match(val_gold_sets, val_predicted)
print(f"validation exact domain-set match: {val_exact:.3f}{selection}{check}")
test_exact = exact_set_match(test_gold_sets, test_predicted)
print(f"test exact domain-set match: {test_exact:.3f}{check}")

print("test passages the composed set got wrong:")
for e, predicted, gold in zip(test_examples, test_predicted, test_gold_sets, strict=True):
    if predicted != gold:
        print(f"  {e.id}: gold {sorted(gold)}  tagged {sorted(predicted)}")

### The confidence gate's own outcomes

In [ ]:
def outcomes_for(chosen_examples, gold_by_domain):
    """Coverage, accuracy and risk of tag_domains's own yes/no outcomes, pooled over every
    domain with jev_cookbook.evaluation.pool_label_outcomes and evaluate_outcomes -- never a
    separate reimplementation of noul >= threshold or the confidence check."""
    accepted_by_domain = {
        d: [all_decisions[e.id][d].outcome != helpers.REVIEW for e in chosen_examples]
        for d in DOMAINS
    }
    tag_by_domain = {d: [all_decisions[e.id][d].tag for e in chosen_examples] for d in DOMAINS}
    accepted, correct, _ = pool_label_outcomes(
        [e.id for e in chosen_examples], DOMAINS, gold_by_domain, accepted_by_domain, tag_by_domain
    )
    return evaluate_outcomes(accepted, correct)


val_outcomes = outcomes_for(val_examples, val_gold_by_domain)
print(f"(passage, domain) pairs on validation: {val_outcomes.n_total}{selection}")
print(
    f"answered: {val_outcomes.n_answered} of {val_outcomes.n_total} "
    f"(coverage {val_outcomes.coverage:.4f}){selection}{check}"
)
print(f"accuracy among those answered: {val_outcomes.accuracy:.4f}{selection}{check}")
print(f"risk among those answered: {val_outcomes.risk:.4f}{selection}{check}")

test_gold_by_domain = {d: [d in test_gold[e.id] for e in test_examples] for d in DOMAINS}
test_outcomes = outcomes_for(test_examples, test_gold_by_domain)
print(f"(passage, domain) pairs on test: {test_outcomes.n_total}{check}")
print(
    f"answered: {test_outcomes.n_answered} of {test_outcomes.n_total} "
    f"(coverage {test_outcomes.coverage:.4f}){check}"
)
print(f"accuracy among those answered: {test_outcomes.accuracy:.4f}{check}")
print(f"risk among those answered: {test_outcomes.risk:.4f}{check}")

# tag_domains's only review branch is the shared confidence gate, so the confidence-only view
# (evaluate_selective, reapplying confidence >= gate itself) must agree with the rule's own
# outcomes view (evaluate_outcomes, above) on every field but threshold. Printed here, not only
# asserted: the two rows below should read identically except for that one field.
test_correct, test_confidence, _ = pool_label_decisions(
    [e.id for e in test_examples], DOMAINS, test_gold_by_domain, test_noul_by_domain, thresholds
)
test_selective = evaluate_selective(test_correct, test_confidence, confidence_gate)
print(
    f"confidence-only view on test: answered {test_selective.n_answered} of "
    f"{test_selective.n_total} (coverage {test_selective.coverage:.4f})  "
    f"accuracy {test_selective.accuracy:.4f}  risk {test_selective.risk:.4f}{check}"
)

This rule's only review branch is the shared confidence gate -- nothing else sends a (passage, domain) pair to review, and nothing lets one through regardless of confidence -- so `evaluate_outcomes` (the rule's own outcomes) and `evaluate_selective` (the confidence-only view) must agree on every field but `threshold`. The two printed rows for `test` above do: both read `answered 42 of 54`, `coverage 0.7778`, `accuracy 0.9524`, `risk 0.0476` -- shown side by side rather than only asserted, which is the habit that pays off directly for a rule with more than one review branch, where the two would *not* agree.

`test` risk, 0.0476, is two of the 42 (passage, domain) pairs the gate answers on `test`: `t19-economics-biomedical-wrong`'s confidently wrong `biomedical` tag (0.84, confidence 0.68) and `t20-economics-computational-wrong`'s confidently wrong `computational` tag (0.84, confidence 0.68). Neither is a low-confidence pair the gate was ever built to catch -- both clear the frozen 0.6400 gate by a comfortable margin -- which is the pooling caveat from "Python's part" showing up exactly where it was predicted to: each domain's threshold sits around 0.8, far from 0.5, so an answer can be `noul_confidence`-confident while still sitting close to the line that decides it, and a gate pooled across all three domains has no way to single out which domain's margin problem is worse than another's. A gate chosen for high coverage on 54 pooled `validation` pairs is a guarantee about those pairs, not about any pair this recipe has not seen, confident or not.

### The access-control test

The use case's own acceptance criterion asks for a test showing that a restricted passage is never sent and never appears in output -- a two-part claim that needs two different kinds of proof. "Never appears in output" is checkable from what is already committed above: no restricted id is a key of `decisions`, `all_decisions`, or `results`, and the `ReviewQueue` printed above never names one, because its loop only ever walks `accessible`. "Never sent" is the harder half, because it is a claim about what the pipeline *would do*, not only about what is sitting in a dict afterwards -- so this reads `backend.sent_states`, the running record `helpers.CountingBackend` has kept since "Setup and run mode" wrapped the one backend this notebook ever builds. Reading that one backend's own history, rather than building a second backend to re-decide everything through, is deliberate: a second backend (`get_backend(...)` called again) would be a second, independent request budget in live mode, and the request count below would then undercount what actually ran. This is the same backend, the same cache, and the same approach `tests/test_helpers.py::test_a_restricted_passage_is_never_sent` takes (and the same test that fails, naming the restricted passage, when `helpers.decide_passages`'s access-control filter is removed).

In [ ]:
# backend.sent_states is the real, running record of every request this notebook has made,
# from the very first decide() call in "One answer up close" onward: the identifying check
# (which restricted passage, if any, leaked) runs BEFORE the plain count below, so a failure
# here would name the passage rather than only reporting a mismatched count.
restricted_states = {e.id: helpers.build_state(e.fields) for e in restricted}
leaked = [eid for eid, st in restricted_states.items() if st in backend.sent_states]
print(f"restricted passages whose state was ever sent: {len(leaked)}{check}")
if leaked:
    print(f"  leaked: {leaked}{check}")

print(f"total requests sent so far in this notebook: {len(backend.sent_states)}{check}")
print(f"accessible candidates (validation + test): {len(accessible)}{check}")
print(f"restricted candidates withheld (validation + test): {len(restricted)}{check}")

restricted_ids = {e.id for e in restricted}
restricted_passage_ids = {e.fields["passage_id"] for e in restricted}
queued_passage_ids = {item.item["passage_id"] for item in review_queue.pending()}
print(
    f"restricted ids present in the decisions output: {sorted(set(decisions) & restricted_ids)}{check}"
)
print(
    f"restricted ids present in the review queue: "
    f"{sorted(queued_passage_ids & restricted_passage_ids)}{check}"
)

Every number above is zero where it should be and matches `len(accessible)` and `len(restricted)` where it should: four restricted candidates were withheld, none of `backend`'s 37 requests so far (36 accessible candidates plus the one demo passage decided individually in "One answer up close") carried a restricted passage's state, and neither the decisions dict nor the review queue names a restricted id, even by its bookkeeping `passage_id` rather than its fixture `id`. This is not a security guarantee -- the filter is a plain Python `if`, not an authentication layer, and nothing here claims to be an access-control system -- but it is a deterministic one: whatever Jev would have said about a restricted passage, Python decided the question before any request existed to ask it.

## What was and was not measured

Measured: that the pipeline runs end to end, and that the per-domain thresholds, the shared confidence gate, the review queue, the access-control filter and the evaluation above behave as designed on 36 accessible, scored passages (plus the 2 demo passages shown earlier, one accessible and one restricted, never scored; and 4 further restricted passages, counted but never decided). Not measured: anything about how Jev performs, how fast it is, or what it costs. Every stored probability in the offline run was written by hand, several deliberately wrong on purpose -- the look-alike passages discussed above -- so every number above describes this fixture set and not a model.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: {len(val_examples)} validation and {len(test_examples)} test examples "
        f"(plus {len(restricted)} restricted, never decided){check}"
    )
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(
        f"N: {len(val_examples)} validation and {len(test_examples)} test examples "
        f"(plus {len(restricted)} restricted, never decided)"
    )

## Next steps

- Lower `min_coverage` in the confidence-cutoff cell above (for example to `0.50`) and re-run from there: the chosen gate rises from 0.6400 to 0.6800, coverage on `validation` falls from 0.7593 to 0.6852 (both already printed in the sweep table above), and the `validation` review queue grows from 13 items to 17.
- Change `t19-economics-biomedical-wrong`'s gold label in `build_fixtures.py` from `("economics",)` to `("economics", "biomedical")` (its stored 0.84 `biomedical` answer would then be *right*, not wrong), run `python build_fixtures.py`, and re-run "Evaluation": `biomedical`'s `test` precision and F1 both rise to 1.0000 (its one false positive becomes a true positive), and `exact_set_match` on `test` rises from 0.722 to 0.778 -- the same kind of mutation `tests/test_helpers.py`'s own consequentiality tests perform on committed rows, done here by hand, on the fixtures themselves.
- Mark `v01-none-clean` restricted in `build_fixtures.py` (its access to `"restricted"`, its spec to `None`) and regenerate: validation's accessible count falls from 18 to 17 and `v01-none-clean` moves from "Access control"'s accessible list to its restricted one, but every `test` number in "Evaluation" is unchanged, because `test`'s evaluation never depended on which `validation` passage happened to be accessible.
- Neighbouring recipes: [`06-multiple-topic-labels`](../06-multiple-topic-labels/), the Level 1 multi-label `Noul` pattern this recipe builds on, and [`26-prompt-injection-screening`](../26-prompt-injection-screening/), another multi-label `Noul` recipe whose composition also routes some outcomes to a simulated queue.